# RetailPulse — Day 1 EDA

Reproducible exploratory analysis of the actual datasets in `data/raw/`: sales, customers, products, and inventory.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
pd.set_option('display.max_columns', None)
RAW=Path('../data/raw')


In [ ]:
sales=pd.read_csv(RAW/'sales.csv')
customers=pd.read_csv(RAW/'customers.csv')
products=pd.read_csv(RAW/'products.csv')
inventory=pd.read_csv(RAW/'inventory.csv')
for n,d in {'sales':sales,'customers':customers,'products':products,'inventory':inventory}.items():
    print(n, d.shape)
    display(d.head())


## Data quality


In [ ]:
datasets={'sales':sales,'customers':customers,'products':products,'inventory':inventory}
quality=[]
for n,d in datasets.items():
    quality.append({'dataset':n,'rows':len(d),'columns':d.shape[1],'duplicates':int(d.duplicated().sum()),'missing_cells':int(d.isna().sum().sum()),'missing_pct':round(d.isna().mean().mean()*100,2)})
display(pd.DataFrame(quality))


In [ ]:
for n,d in datasets.items():
    print('\n',n)
    display(d.isna().sum().sort_values(ascending=False).to_frame('missing'))


## Type normalization and derived metrics


In [ ]:
sales['date']=pd.to_datetime(sales['date'],errors='coerce')
customers['registration_date']=pd.to_datetime(customers['registration_date'],errors='coerce')
inventory['date']=pd.to_datetime(inventory['date'],errors='coerce')
sales['gross_amount']=sales['quantity']*sales['unit_price']
sales['discount_amount']=sales['gross_amount']*sales['discount']/100
sales['calculated_total']=sales['gross_amount']-sales['discount_amount']
sales['calculation_difference']=sales['total_amount']-sales['calculated_total']
products['unit_margin']=products['selling_price']-products['unit_cost']
products['margin_pct']=products['unit_margin']/products['selling_price']*100


## Sales analysis


In [ ]:
display(sales[['quantity','unit_price','discount','total_amount']].describe().T)
print('Date range:',sales.date.min(),'to',sales.date.max())
print('Transactions:',sales.transaction_id.nunique())
print('Customers in sales:',sales.customer_id.nunique())
print('Products in sales:',sales.product_id.nunique())
print('Total revenue:',sales.total_amount.sum())
print('Total quantity:',sales.quantity.sum())


In [ ]:
fig,ax=plt.subplots(2,2,figsize=(14,9))
sns.histplot(sales.total_amount,bins=50,ax=ax[0,0]); ax[0,0].set_title('Transaction Revenue')
sns.histplot(sales.quantity,bins=50,ax=ax[0,1]); ax[0,1].set_title('Quantity')
sns.histplot(sales.discount,bins=50,ax=ax[1,0]); ax[1,0].set_title('Discount %')
sns.histplot(sales.unit_price,bins=50,ax=ax[1,1]); ax[1,1].set_title('Unit Price')
plt.tight_layout()


In [ ]:
monthly=sales.assign(month=sales.date.dt.to_period('M').astype(str)).groupby('month',as_index=False).agg(revenue=('total_amount','sum'),quantity=('quantity','sum'),transactions=('transaction_id','nunique'))
display(monthly)
plt.figure(figsize=(14,5)); plt.plot(monthly.month,monthly.revenue,marker='o'); plt.xticks(rotation=60); plt.title('Monthly Revenue'); plt.tight_layout()


## Customer analysis


In [ ]:
display(customers.age.describe().to_frame('age'))
display(customers.gender.value_counts(dropna=False).to_frame('customers'))
display(customers.city.value_counts().head(15).to_frame('customers'))
customer_sales=sales.groupby('customer_id').agg(transactions=('transaction_id','nunique'),quantity=('quantity','sum'),revenue=('total_amount','sum'),avg_transaction=('total_amount','mean'),last_purchase=('date','max')).reset_index()
display(customer_sales.sort_values('revenue',ascending=False).head(20))


## Product and category analysis


In [ ]:
cat=sales.merge(products,on='product_id',how='left').groupby('category',as_index=False).agg(revenue=('total_amount','sum'),quantity=('quantity','sum'),transactions=('transaction_id','nunique')).sort_values('revenue',ascending=False)
display(cat)
plt.figure(figsize=(12,6)); sns.barplot(data=cat,x='revenue',y='category'); plt.title('Revenue by Category'); plt.tight_layout()
product_perf=sales.merge(products,on='product_id',how='left').groupby(['product_id','product_name','category'],as_index=False).agg(revenue=('total_amount','sum'),quantity=('quantity','sum')).sort_values('revenue',ascending=False)
display(product_perf.head(20))


## Inventory analysis


In [ ]:
inventory['stock_gap']=inventory.stock_quantity-inventory.reorder_level
inventory['below_reorder']=inventory.stock_quantity<inventory.reorder_level
display(inventory[['stock_quantity','reorder_level','lead_time']].describe().T)
print('Below reorder:',inventory.below_reorder.sum(),'/',len(inventory),'=',round(inventory.below_reorder.mean()*100,2),'%')
display(inventory.sort_values('stock_gap').head(20))


## Correlation and integrity checks


In [ ]:
plt.figure(figsize=(10,7)); sns.heatmap(sales.select_dtypes(include=np.number).corr(),annot=True,fmt='.2f',cmap='coolwarm'); plt.title('Sales Numeric Correlation'); plt.tight_layout()
print('Sales -> customer missing matches:',(~sales.customer_id.astype(str).isin(customers.customer_id.astype(str))).sum())
print('Sales -> product missing matches:',(~sales.product_id.isin(products.product_id)).sum())
print('Inventory -> product missing matches:',(~inventory.product_id.isin(products.product_id)).sum())


## Day-1 conclusion

Use the notebook outputs as the factual basis for Day 2 cleaning. Do not hard-code findings; first validate missing values, duplicates, referential integrity, outliers, and the revenue calculation.
